# MEN 4338 · Week 4 — Directional variograms and anisotropy

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/men4338/notebooks/MEN4338_W4_anisotropy_directional_variograms.ipynb)

**IUGB · BSc Mining Engineering · Geostatistics (lecture & lab)** · Instructor: Laurent Kouadio

Gold in a shear zone, sand in a channel, a weathering front: geology is rarely the same in every direction.
If grades stay similar over longer distances along one direction, the variogram rises more slowly in that
direction: this is **anisotropy**. Missing it leads to estimates smeared across the structure.

**What you will practise:** azimuths and the angular tolerance · directional variograms · the variogram map ·
major and minor directions · the anisotropy ratio.

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/men4338/'
FILES = [('data/bassam_samples.csv', 'bassam_samples.csv'), ('tests/w4/q1.py', 'tests/q1.py'), ('tests/w4/q2.py', 'tests/q2.py'), ('tests/w4/q3.py', 'tests/q3.py'), ('tests/w4/q4.py', 'tests/q4.py'), ('tests/w4/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

In [ ]:
# Toolbox from earlier weeks: run this cell.
import numpy as np

def exp_variogram(x, y, v, lag, nlags, azimuth=None, atol=22.5):
    """Experimental semivariogram. Lag class k (1..nlags) holds the pairs with |d - k*lag| < lag/2.
    With an azimuth (degrees clockwise from north), only pairs within +/- atol of that direction are kept.
    Returns mean distance, semivariance and number of pairs of each class (NaN where a class is empty)."""
    x, y, v = (np.asarray(a, float) for a in (x, y, v))
    i, j = np.triu_indices(len(v), 1)
    dx, dy = x[j] - x[i], y[j] - y[i]
    d = np.hypot(dx, dy)
    keep = np.ones(len(d), bool)
    if azimuth is not None:
        az = np.degrees(np.arctan2(dx, dy)) % 180
        diff = np.abs(az - azimuth % 180)
        keep = np.minimum(diff, 180 - diff) <= atol
    sq = 0.5 * (v[j] - v[i]) ** 2
    hm, g, n = np.full(nlags, np.nan), np.full(nlags, np.nan), np.zeros(nlags, int)
    for k in range(1, nlags + 1):
        m = keep & (np.abs(d - k * lag) < lag / 2)
        n[k - 1] = m.sum()
        if n[k - 1]:
            hm[k - 1], g[k - 1] = d[m].mean(), sq[m].mean()
    return hm, g, n

The toolbox cell above contains a complete `exp_variogram` (your week 3 function, extended with a direction).
This week you build the directional part yourself.

## 1. Azimuths

In mining, directions are **azimuths**: degrees clockwise from north. For a pair separated by
$(\Delta x, \Delta y)$ the azimuth is `np.degrees(np.arctan2(dx, dy))`. Note the order: `dx` first,
because we measure from north (the $y$ axis), not from east. A pair has no "front": 30° and 210° are the same
direction, so azimuths are folded into $[0, 180)$.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

samples = pd.read_csv("bassam_samples.csv")
grid = samples[samples["campaign"] == "grid"]
x, y, au = grid["x"].to_numpy(), grid["y"].to_numpy(), grid["au_gpt"].to_numpy()
print(np.degrees(np.arctan2(1, 1)), np.degrees(np.arctan2(1, -1)) % 180)   # NE = 45, SE = 135

## 2. Directional variograms

A directional variogram keeps only the pairs whose azimuth is within an **angular tolerance** (typically
±22.5°) of the chosen direction. Too narrow and there are too few pairs; too wide and the directions mix.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
h, g, n = exp_variogram(x, y, au, 50, 10)
ax.plot(h, g, "k.-", label="omnidirectional")
for az in [0, 90]:
    h, g, n = exp_variogram(x, y, au, 50, 10, azimuth=az, atol=22.5)
    ax.plot(h, g, "o-", label=f"N{az:03d}")
ax.axhline(au.var(ddof=1), ls="--", color="grey")
ax.set_xlabel("h (m)"); ax.set_ylabel("γ(h)"); ax.legend(); plt.show()

North–south and east–west look alike here. That does not mean there is no anisotropy: the structure
may be oblique. We need to test more directions, or look at all of them at once with a **variogram map**.

## 3. Geometric anisotropy

When the sill is the same in every direction but the range changes, the anisotropy is **geometric**. It is
described by the azimuth of the **major** direction (longest range), the major range $a_{max}$ and the minor
range $a_{min}$ (perpendicular). The ratio $a_{min}/a_{max}$ is the anisotropy ratio. Plotted around a circle,
the ranges draw an **ellipse**.

---
## Exercises (grid holes: `x`, `y`, `au`)

### Q1. Azimuth of a pair

Write `azimuth_of(dx, dy)` returning the azimuth in degrees in $[0, 180)$. It must work on numbers and arrays.

In [ ]:
def azimuth_of(dx, dy):
    ...

azimuth_of(np.array([0., 1, 1, -1]), np.array([1., 1, 0, 1]))   # expect 0, 45, 90, 135

In [ ]:
grader.check("q1")

### Q2. Your directional variogram

Write `dir_variogram(x, y, v, lag, nlags, azimuth, atol)` returning `(h, gamma, npairs)` like `exp_variogram`,
but keeping only the pairs whose azimuth is within `atol` degrees of `azimuth`.
Careful near north: a pair at 178° is only 2° from 0°.

In [ ]:
def dir_variogram(x, y, v, lag, nlags, azimuth, atol):
    ...

dir_variogram(x, y, au, 50, 10, 60, 22.5)[1]

In [ ]:
grader.check("q2")

### Q3. Four directions

Build the dictionary `dir_gammas` mapping the azimuths 0, 45, 90 and 135 to their semivariance arrays (lag
50 m, 10 lags, tolerance 22.5°). Plot the four variograms on one figure.

In [ ]:
dir_gammas = {}
...

fig, ax = plt.subplots(figsize=(7, 4))
for az, g in dir_gammas.items():
    ax.plot(np.arange(1, 11) * 50, g, "o-", label=f"N{az:03d}")
ax.legend(); ax.set_xlabel("h (m)"); plt.show()

In [ ]:
grader.check("q3")

### Q4. Major and minor directions

The major direction is the one whose variogram rises the slowest. Use the mean of the first 4 lags of each
direction: store the azimuth with the smallest mean in `major_az`, the perpendicular direction in `minor_az`,
and `continuity_ratio` = (mean of the first 4 lags along `minor_az`) / (same along `major_az`).
Does `major_az` agree with the elongation of the rich zone on the week 1 map?

In [ ]:
major_az = ...
minor_az = ...
continuity_ratio = ...
major_az, minor_az, continuity_ratio

In [ ]:
grader.check("q4")

### Q5. The variogram map

Compute the variogram map on a 12 × 12 grid of 50 m cells covering separations from −300 to +300 m in
$\Delta x$ and $\Delta y$. Use **both** orientations of each pair, $(\Delta x, \Delta y)$ and
$(-\Delta x, -\Delta y)$. Cell `[row, col]` with `row = floor((dy + 300) / 50)`, `col = floor((dx + 300) / 50)`
holds the mean of $\frac12(z_i - z_j)^2$ of its pairs (NaN if empty). Store it in `vmap` and draw it in
`fig_vm, ax_vm` with `imshow(vmap, origin="lower", extent=[-300, 300, -300, 300])`.
The low values (dark) draw an ellipse elongated along the major direction.

*Hint:* `np.add.at(sums, (rows, cols), values)` adds values into a 2D array cell by cell.

In [ ]:
vmap = ...

fig_vm, ax_vm = plt.subplots(figsize=(5.5, 5))
...
plt.show()

In [ ]:
grader.check("q5")

### Going further (not checked)

* Repeat Q3 with 8 directions (every 22.5°) and an angular tolerance of 11.25°. Is the major direction
  sharper?
* For the major and minor directions, read the distance where each variogram reaches the sill: what is
  the anisotropy ratio $a_{min}/a_{max}$?

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()